# 05 — Emotion vectors: geometry + how emotion loads into stories

**Project:** [EmoVecLLM](https://github.com/drgzkr/EmoVecLLM) — replicating Anthropic 2026 emotion vectors on open-source LLMs.

The figures notebook for **nb04**. It reads nb04's three artefacts and draws preliminary figures — no model, no GPU, pure `numpy`/`matplotlib`/`sklearn`, so it runs on whatever is already on disk (including a partial **demo** run).

**Figures**

1. **Availability** — segments per emotion + neutral baseline coverage (the health check on a partial run).
2. **Geometry** — emotion vectors in PCA space coloured by k-means cluster, and the emotion×emotion cosine-similarity matrix ordered by cluster.
3. **Loading curves** — the headline: each emotion's average vector **projected onto the word-by-word cumulative feature timeseries** of its stories, showing the emotion direction being *loaded* as the model reads. We contrast the **matched** emotion vector against **mismatched** ones to show specificity.

> Everything is keyed off whatever `features/{spec_hash}/{target_model}/` directory nb04 last wrote (auto-discovered, or set `EMOVEC_FEATURES_DIR`).

## 0 — Setup

In [ ]:
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "scikit-learn", "matplotlib"], check=False)


In [ ]:
import json, os, re
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

def _env(name, default):
    v = os.environ.get(name)
    return v if v not in (None, "") else default

def _env_int(name, default):
    v = os.environ.get(name)
    return int(v) if v not in (None, "") else default

def _env_flag(name, default):
    v = os.environ.get(name)
    return default if v is None else v.strip().lower() in ("1", "true", "yes", "on")

def _unit(v, axis=-1, eps=1e-12):
    return v / (np.linalg.norm(v, axis=axis, keepdims=True) + eps)

MOUNT_DRIVE = _env_flag("EMOVEC_MOUNT_DRIVE", IN_COLAB)
if os.environ.get("EMOVEC_WORK_DIR"):
    WORK_DIR = Path(os.environ["EMOVEC_WORK_DIR"]).expanduser()
elif IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/EmoVecLLM")
elif IN_COLAB:
    WORK_DIR = Path("/content/EmoVecLLM")
else:
    WORK_DIR = Path("..").resolve()

DATA_DIR = WORK_DIR / "data" / "processed"
FIG_DIR = WORK_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 10, "axes.labelsize": 9,
})
print(f"WORK_DIR = {WORK_DIR}")


## 1 — Locate nb04's features

Auto-discovers the most recently written `features/{spec_hash}/{target_model}/` (most segments), or honours `EMOVEC_FEATURES_DIR`.

In [ ]:
FEATURES_DIR = _env("EMOVEC_FEATURES_DIR", "")
if FEATURES_DIR:
    feat_dir = Path(FEATURES_DIR).expanduser()
else:
    manifests = sorted((DATA_DIR / "features").rglob("features_manifest.json"))
    assert manifests, (f"no features under {DATA_DIR/'features'} — run nb04 first, "
                       f"or set EMOVEC_FEATURES_DIR")
    def _nseg(p):
        try:
            return json.loads(p.read_text()).get("n_segments", 0)
        except Exception:
            return 0
    feat_dir = max(manifests, key=_nseg).parent
manifest = json.loads((feat_dir / "features_manifest.json").read_text())
print(f"features dir : {feat_dir}")
print(json.dumps(manifest, indent=2))


In [ ]:
seg = np.load(feat_dir / "segment_features.npz", allow_pickle=True)
vec = np.load(feat_dir / "emotion_vectors.npz", allow_pickle=True)
cum = np.load(feat_dir / "cumulative_timeseries.npz", allow_pickle=True)

X            = seg["X"]                       # (n_seg, n_layers, d)
seg_emotions = seg["emotions"]
seg_kinds    = seg["kinds"]

V            = vec["vectors"]                 # (E, n_layers, d)
emotions     = list(vec["emotions"])
cluster_lab  = vec["cluster_labels"]
CLUSTER_L    = int(vec["cluster_layer"])
CUM_L        = int(vec["cum_layer"])
baseline     = vec["baseline_mean"]          # (n_layers, d)
baseline_mode = str(vec["baseline_mode"])
n_per_emotion = vec["n_per_emotion"]
TARGET_MODEL = str(vec["target_model"])

cum_curves   = cum["cum"]                     # object array of (seq, n_sel_layers, d)
cum_emotions = cum["cum_emotions"]
cum_kinds    = cum["cum_kinds"]
cum_tokens   = cum["cum_tokens"]
cum_layer_idx  = [int(x) for x in cum["cum_layer_indices"]]
cum_proj_layer = int(cum["cum_proj_layer"])
PROJ_POS       = cum_layer_idx.index(cum_proj_layer)   # which slice of axis-1 to project

E, n_layers, d_model = V.shape
DEMO = manifest.get("demo", False)
print(f"target={TARGET_MODEL}  emotions={E}  layers={n_layers}  d={d_model}")
print(f"baseline={baseline_mode}  cluster_layer={CLUSTER_L}  proj_layer={CUM_L}")
print(f"per-step timeseries: {len(cum_curves)} stories, stored layers={cum_layer_idx} "
      f"(project at layer {cum_proj_layer})"
      + ("   [DEMO run — figures are illustrative]" if DEMO else ""))


## 2 — Figure 1 · What's available

Segments per emotion (sorted), with the neutral-baseline count called out. On a partial run this immediately shows which emotions are under-sampled and whether a baseline exists at all.

In [ ]:
order = np.argsort(n_per_emotion)[::-1]
emos_sorted = [emotions[i] for i in order]
counts_sorted = n_per_emotion[order]
n_neutral = int(np.sum(np.isin(seg_kinds, ["neutral_dialogue", "neutral_story"])))

fig, ax = plt.subplots(figsize=(max(6, 0.16 * E + 2), 3.4))
ax.bar(range(E), counts_sorted, color="#3b6ea5", width=0.9)
ax.axhline(np.mean(counts_sorted), color="#d1495b", lw=1, ls="--",
           label=f"mean {np.mean(counts_sorted):.1f}")
ax.set_xlim(-0.5, E - 0.5)
ax.set_xlabel(f"emotion (n={E}, sorted by coverage)")
ax.set_ylabel("segments")
ax.set_title(f"Story coverage per emotion · target={TARGET_MODEL} · "
             f"neutral baseline: {n_neutral} segs ({baseline_mode})")
# Label only the extremes to avoid clutter.
show = list(range(min(6, E))) + list(range(max(0, E - 6), E))
ax.set_xticks(show)
ax.set_xticklabels([emos_sorted[i] for i in show], rotation=45, ha="right", fontsize=6)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "nb05_fig1_coverage.png", bbox_inches="tight")
plt.show()
print(f"total segments {int(np.sum(n_per_emotion))} across {E} emotions; "
      f"neutral {n_neutral}")


## 3 — Figure 2 · Emotion-vector geometry

Left: the emotion vectors at layer `CLUSTER_L` projected to 2D (PCA), coloured by k-means cluster — the open-source echo of Anthropic's valence/arousal circumplex (nb07 will align PC1/PC2 to Warriner norms). Right: the emotion×emotion **cosine-similarity** matrix, rows/cols ordered by cluster so blocks on the diagonal = coherent emotion families.

In [ ]:
from sklearn.decomposition import PCA

Vc = V[:, CLUSTER_L, :]                       # (E, d)
Vu = _unit(Vc)
proj = PCA(n_components=2).fit(Vu)
P2 = proj.transform(Vu)
evr = proj.explained_variance_ratio_

# Cosine matrix ordered by cluster.
ord_by_cluster = np.argsort(cluster_lab, kind="stable")
S = (_unit(Vc) @ _unit(Vc).T)[np.ix_(ord_by_cluster, ord_by_cluster)]

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5.4),
                               gridspec_kw={"width_ratios": [1.15, 1]})

k = int(cluster_lab.max()) + 1
cmap = plt.cm.tab10 if k <= 10 else plt.cm.tab20
for c in range(k):
    m = cluster_lab == c
    axL.scatter(P2[m, 0], P2[m, 1], s=42, color=cmap(c % cmap.N),
                edgecolor="white", linewidth=0.5, label=f"c{c}", zorder=2)
# Light labels (sparse if many emotions).
lab_idx = range(E) if E <= 40 else np.argsort(np.linalg.norm(P2, axis=1))[::-1][:40]
for i in lab_idx:
    axL.annotate(emotions[i], (P2[i, 0], P2[i, 1]), fontsize=6, alpha=0.8,
                 xytext=(3, 2), textcoords="offset points")
axL.set_xlabel(f"PC1 ({evr[0]*100:.1f}%)")
axL.set_ylabel(f"PC2 ({evr[1]*100:.1f}%)")
axL.set_title(f"Emotion vectors (layer {CLUSTER_L}) — PCA, coloured by cluster")
axL.legend(frameon=False, fontsize=7, ncol=2, loc="best")

vmax = float(np.percentile(np.abs(S - np.eye(S.shape[0])), 99)) or 1.0
im = axR.imshow(S, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axR.set_title(f"Cosine similarity (ordered by cluster)")
# Cluster boundary lines.
bounds = np.where(np.diff(cluster_lab[ord_by_cluster]) != 0)[0] + 0.5
for b in bounds:
    axR.axhline(b, color="k", lw=0.4); axR.axvline(b, color="k", lw=0.4)
if E <= 40:
    axR.set_xticks(range(E)); axR.set_yticks(range(E))
    axR.set_xticklabels([emotions[i] for i in ord_by_cluster], rotation=90, fontsize=5)
    axR.set_yticklabels([emotions[i] for i in ord_by_cluster], fontsize=5)
fig.colorbar(im, ax=axR, shrink=0.8, label="cosine")
fig.tight_layout()
fig.savefig(FIG_DIR / "nb05_fig2_geometry.png", bbox_inches="tight")
plt.show()


## 4 — Figure 3 · How emotion is *loaded* into a story

The per-step timeseries `f[t]` is the residual after the model has read the prefix `tokens[:t+1]` — its state at each point in the story (the cumulation is done by the model's attention). Project it (centred on the baseline, at the projection layer) onto the **unit emotion vector** to get a scalar per token:

$$\text{load}_e(t) = \big(\,f(t) - \text{baseline}\,\big)\cdot \hat{v}_e$$

For a handful of example stories we plot the load along their *own* emotion (**matched**) against the mean of the *other* emotions' vectors (**mismatched**). A matched curve that climbs above mismatched as the story unfolds is the signature of the emotion direction being progressively written into the residual stream.

In [ ]:
emo_index = {e: i for i, e in enumerate(emotions)}
vhat = _unit(V[:, CUM_L, :])                  # (E, d) unit emotion dirs at cum layer
base_c = baseline[CUM_L]                       # (d,)

def load_curve(feat_seq, e_idx):
    'feat_seq (seq, n_sel_layers, d): project the projection-layer residual onto emotion e_idx.'
    f = feat_seq[:, PROJ_POS, :].astype(np.float64)        # (seq, d) at projection layer
    return (f - base_c) @ vhat[e_idx]

# Pick example emotion stories that have a saved timeseries (spread over emotions).
ex = []
seen = set()
for j in range(len(cum_curves)):
    e = cum_emotions[j]
    if cum_kinds[j] == "emotion_story" and e in emo_index and e not in seen:
        ex.append(j); seen.add(e)
    if len(ex) >= 6:
        break

if ex:
    ncol = min(3, len(ex))
    nrow = int(np.ceil(len(ex) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 3.1 * nrow),
                             squeeze=False)
    for ax in axes.flat:
        ax.set_visible(False)
    for ax, j in zip(axes.flat, ex):
        ax.set_visible(True)
        e = cum_emotions[j]; ei = emo_index[e]
        seqd = cum_curves[j]
        matched = load_curve(seqd, ei)
        others = [k for k in range(E) if k != ei]
        mismatched = np.mean([load_curve(seqd, k) for k in others], axis=0)
        t = np.arange(len(matched))
        ax.plot(t, matched, color="#d1495b", lw=1.8, label=f"matched: {e}")
        ax.plot(t, mismatched, color="#8d99ae", lw=1.2, ls="--", label="mean other")
        ax.axhline(0, color="k", lw=0.5, alpha=0.4)
        ax.set_title(f"{e}", fontsize=9)
        ax.set_xlabel("token position")
        ax.set_ylabel("projection")
        ax.legend(frameon=False, fontsize=7)
    fig.suptitle(f"Emotion loading along the story (layer {CUM_L}, target={TARGET_MODEL})",
                 y=1.01)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "nb05_fig3_loading_examples.png", bbox_inches="tight")
    plt.show()
else:
    print("no emotion-story cumulative timeseries found — re-run nb04 with stories present.")


### 4b — Aggregate loading: matched vs mismatched

Each story's load curve is resampled onto a common fractional-position axis (0 → start, 1 → end) and averaged. If the emotion direction is genuinely written during reading, the **matched** mean rises above the **mismatched** mean and the gap widens toward the end.

In [ ]:
def resample(y, n=50):
    y = np.asarray(y, dtype=np.float64)
    if len(y) < 2:
        return np.full(n, y[0] if len(y) else 0.0)
    xp = np.linspace(0, 1, len(y))
    return np.interp(np.linspace(0, 1, n), xp, y)

N = 50
matched_all, mismatched_all = [], []
for j in range(len(cum_curves)):
    e = cum_emotions[j]
    if cum_kinds[j] != "emotion_story" or e not in emo_index:
        continue
    ei = emo_index[e]; seqd = cum_curves[j]
    matched_all.append(resample(load_curve(seqd, ei), N))
    others = [k for k in range(E) if k != ei]
    mismatched_all.append(resample(np.mean([load_curve(seqd, k) for k in others], axis=0), N))

if matched_all:
    M = np.stack(matched_all); MM = np.stack(mismatched_all)
    x = np.linspace(0, 1, N)
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    for arr, col, lab in [(M, "#d1495b", "matched emotion"),
                          (MM, "#8d99ae", "mismatched (mean other)")]:
        mu = arr.mean(0); se = arr.std(0) / np.sqrt(len(arr))
        ax.plot(x, mu, color=col, lw=2, label=lab)
        ax.fill_between(x, mu - se, mu + se, color=col, alpha=0.2)
    ax.axhline(0, color="k", lw=0.5, alpha=0.4)
    ax.set_xlabel("fractional story position (0=start, 1=end)")
    ax.set_ylabel("projection onto emotion vector")
    ax.set_title(f"Mean emotion loading ({len(matched_all)} stories, layer {CUM_L})")
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "nb05_fig3b_loading_aggregate.png", bbox_inches="tight")
    plt.show()
    gap = (M[:, -1] - MM[:, -1])
    print(f"end-of-story matched − mismatched: mean {gap.mean():+.3f} "
          f"(matched > mismatched in {100*np.mean(gap>0):.0f}% of stories)")
else:
    print("no aggregate curves available yet.")


### 4c — The raw token × feature timeseries (one layer, one story)

This is the underlying object the loading curve is computed *from*: nb04's per-step array `cum[j]` has shape `(seq, n_sel_layers, d_model)`. Pick **one stored layer** and **one story** and you get a `(seq, d_model)` matrix — the residual stream at every feature, at every step of cumulatively reading the story. Plotted as a heatmap with **token position on x** (left→right = the model reading the story) and **feature on y**.

Each feature is z-scored across tokens so the temporal pattern is visible (raw residual norms grow with depth and dwarf the dynamics). `LAYER_TO_PLOT` must be one of the stored `cum_layer_idx`; `EXAMPLE` indexes the saved timeseries. For large `d_model` (7–8B models) the heatmap is tall — flip `TOP_FEATURES` on to show only the highest-variance features.

In [ ]:
LAYER_TO_PLOT = cum_proj_layer        # any layer in cum_layer_idx
EXAMPLE       = 0                      # index into the stored per-step timeseries
TOP_FEATURES  = 0                     # 0 = all features; else show this many top-variance

if not len(cum_curves):
    print("no per-step timeseries saved — run nb04 with stories present.")
elif LAYER_TO_PLOT not in cum_layer_idx:
    print(f"layer {LAYER_TO_PLOT} not stored; available: {cum_layer_idx}. "
          f"Re-run nb04 with EMOVEC_CUM_LAYERS including it (e.g. 'all').")
else:
    lp = cum_layer_idx.index(LAYER_TO_PLOT)
    feat = np.asarray(cum_curves[EXAMPLE][:, lp, :], dtype=np.float64)   # (seq, d)
    toks = [str(t).replace("Ġ", " ").replace("Ċ", "\\n").strip()
            for t in list(cum_tokens[EXAMPLE])][:feat.shape[0]]
    # z-score each feature across tokens.
    fz = (feat - feat.mean(0)) / (feat.std(0) + 1e-12)
    if TOP_FEATURES and TOP_FEATURES < fz.shape[1]:
        keep = np.argsort(feat.var(0))[::-1][:TOP_FEATURES]
        fz = fz[:, keep]
        ylab = f"top-{TOP_FEATURES} features (by variance)"
    else:
        ylab = f"feature (d_model={feat.shape[1]})"

    seq = fz.shape[0]
    fig, ax = plt.subplots(figsize=(max(6, 0.16 * seq + 2), 5.2))
    vmax = float(np.percentile(np.abs(fz), 99)) or 1.0
    im = ax.imshow(fz.T, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                   interpolation="nearest", origin="lower")
    ax.set_xlabel("token position  (cumulative reading →)")
    ax.set_ylabel(ylab)
    ax.set_title(f"Per-step residual — story {EXAMPLE} "
                 f"({cum_emotions[EXAMPLE]}), layer {LAYER_TO_PLOT}, target={TARGET_MODEL}")
    if seq <= 60:
        ax.set_xticks(range(seq))
        ax.set_xticklabels(toks, rotation=90, fontsize=6)
    fig.colorbar(im, ax=ax, shrink=0.8, label="z-scored activation")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "nb05_fig4_token_feature.png", bbox_inches="tight")
    plt.show()
    print(f"plotted (seq={seq}, d={feat.shape[1]}) at layer {LAYER_TO_PLOT}; "
          f"emotion={cum_emotions[EXAMPLE]}")


## 5 — Reading these figures + the neutral-baseline question

**Demo caveat.** With `EMOVEC_DEMO=1` and a tiny target (`gpt2`), expect noisy geometry and weak loading separation — the *plumbing* is what's being validated, not the science. Scale up via `EMOVEC_DEMO=0`, a real target (`EleutherAI/pythia-1.4b` → the headline 7–8B models), and full story coverage.

**The neutral baseline shapes every panel here.** The loading curves are projections onto baseline-centred vectors, so what counts as "neutral" sets the zero. The active baseline mode is printed in §1 (`baseline_mode`). Two regimes (see nb04 §6 for the switch):

- **Neutral dialogue** (Anthropic): form-mismatched to our story prose, so a story-vs-dialogue *style* axis contaminates the vectors → prefer `EMOVEC_BASELINE=project_pcs`, and read the loading curves knowing some of the matched rise is shared "narrative" style, not emotion.
- **Neutral stories** (our likely choice): style-matched, so `neutral_mean` should clean the space with less PC surgery, *but* watch that we don't subtract genuine shared narrative content. The matched-vs-mismatched gap in §4b is the quickest read on whether the baseline is doing the right thing.

**Next:** nb06 (held-out scoring on EmoBank/GoEmotions) and nb07 (PCA → Warriner valence/arousal alignment, k-means labelling). Both consume the same `features/` artefacts written by nb04.

In [ ]:
print("figures written to:", FIG_DIR)
for p in sorted(FIG_DIR.glob("nb05_*.png")):
    print("  ", p.name)
